# PGM Forcing Generator

This notebook combines:
- Task 1: standardized model-ready DFS outputs
- Task 2: data-download-tool repository access
- New: DFS0 + gridcode to DFS2 grid series using DHI dfs02todfs2

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import shutil
import subprocess
import sys

import pandas as pd

REPO_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = REPO_ROOT.joinpath("src")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def _run_command(cmd: list[str], error_context: str) -> None:
    proc = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            f"{error_context}\nCommand: {' '.join(cmd)}\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )


def _ensure_pip_available(python_exe: str) -> None:
    has_pip = subprocess.run(
        [python_exe, "-m", "pip", "--version"],
        check=False,
        capture_output=True,
        text=True,
    )
    if has_pip.returncode == 0:
        return

    _run_command(
        [python_exe, "-m", "ensurepip", "--upgrade"],
        "pip is not available in this Python environment and ensurepip failed.",
    )


def install_local_data_downloader_editable(local_repo: Path, python_exe: str) -> None:
    _ensure_pip_available(python_exe)
    _run_command(
        [python_exe, "-m", "pip", "install", "-e", str(local_repo)],
        "Editable install of local data-downloader failed.",
    )


# Optional: point this to a local data-download-tool clone for editable development.
# Example: Path(r"C:\Users\jaan\Repos\phishes-pdp\data-download-tool")
LOCAL_DATA_DOWNLOADER_REPO = Path(r"C:\Users\jaan\Repos\phishes-pdp\data-download-tool")

if LOCAL_DATA_DOWNLOADER_REPO:
    local_repo = Path(LOCAL_DATA_DOWNLOADER_REPO).expanduser().resolve()
    if not local_repo.exists():
        raise FileNotFoundError(f"Local data-downloader repo not found: {local_repo}")
    if not local_repo.joinpath("pyproject.toml").exists():
        raise FileNotFoundError(
            f"No pyproject.toml found in local data-downloader repo: {local_repo}"
        )

    install_local_data_downloader_editable(
        local_repo=local_repo, python_exe=sys.executable
    )

    local_src = local_repo.joinpath("src")
    if local_src.exists() and str(local_src) not in sys.path:
        sys.path.insert(0, str(local_src))

from plant_growth_module import forcing_repository  # noqa: E402

# Trigger data-downloader import and compatibility fallback logic once at startup.
forcing_repository.get_data_downloader_classes()

print("Repo root:", REPO_ROOT)
if LOCAL_DATA_DOWNLOADER_REPO:
    print("data-downloader source:", Path(LOCAL_DATA_DOWNLOADER_REPO).resolve())
    print("data-downloader install mode: editable (-e)")
else:
    print("data-downloader source: environment-installed package")
print("data-downloader package import: OK")

In [ ]:
TIME_RANGE = ("2015-01-01", "2015-01-31")
CATCHMENT_SHP = None
MODEL_EXTENT = [10.0, 55.0, 11.0, 56.0]
EXTENT_CRS = "EPSG:4326"

# Store workflow artifacts under .github/plans following repo instruction preference.
OUTPUT_BASE = REPO_ROOT.joinpath(".github", "plans", "task3_forcing_pipeline")
STANDARDIZED_DIR = OUTPUT_BASE.joinpath("pgm_forcings")

DFS02TODFS2_EXE = Path(
    r"C:\Program Files (x86)\DHI\MIKE Zero\2025\bin\x64\dfs02todfs2.dll"
)
PFS_WORK_DIR = OUTPUT_BASE.joinpath("dfs02todfs2_jobs")
GRID_CODE_DFS2 = REPO_ROOT.joinpath(
    "sample_data", "plant_growth_module", "landuse1_clo.dfs2"
)

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
STANDARDIZED_DIR.mkdir(parents=True, exist_ok=True)
PFS_WORK_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
@dataclass(frozen=True)
class PGMForcingSpec:
    key: str
    category: str
    subcategory: str
    display_name: str
    item_name: str
    item_unit: str
    data_value_type: str
    source_variable: str
    output_filename: str


catalog = forcing_repository.load_data_downloader_catalog()
climate_catalog = catalog.get("climate", {})

# Use all climate entries from dataset_catalog.yaml.
PGM_FORCING_KEYS = sorted(climate_catalog.keys())

FORCING_LIBRARY = []
missing_keys = []

for subcategory in PGM_FORCING_KEYS:
    meta = climate_catalog.get(subcategory)
    if meta is None:
        missing_keys.append(subcategory)
        continue

    output_filename = Path(meta["path"]).with_suffix(".dfs2").name
    FORCING_LIBRARY.append(
        PGMForcingSpec(
            key=subcategory,
            category="climate",
            subcategory=subcategory,
            display_name=meta.get(
                "display_name", subcategory.replace("_", " ").title()
            ),
            item_name=meta.get("variable", ""),
            item_unit=meta.get("eumunit", ""),
            data_value_type=meta.get("data_value_type", ""),
            source_variable=meta.get("variable", ""),
            output_filename=output_filename,
        )
    )

if missing_keys:
    print("Missing climate catalog entries:", ", ".join(missing_keys))

pd.DataFrame([f.__dict__ for f in FORCING_LIBRARY])

In [ ]:
PDPDataDownloader, create_catchment_from_extent = (
    forcing_repository.get_data_downloader_classes()
)
catalog = forcing_repository.load_data_downloader_catalog()
climate_catalog = catalog.get("climate", {})

if CATCHMENT_SHP:
    catchment_input = Path(CATCHMENT_SHP)
else:
    catchment_input = create_catchment_from_extent(extent=MODEL_EXTENT, crs=EXTENT_CRS)

downloader = PDPDataDownloader(
    catchment=catchment_input,
    output_base=OUTPUT_BASE,
    output_format="dfs2",
    buffer_cells=1,
    mask_on_catchment=True,
)

availability = []
for spec in FORCING_LIBRARY:
    availability.append(
        {
            "forcing_key": spec.key,
            "subcategory": spec.subcategory,
            "available_in_catalog": spec.subcategory in climate_catalog,
        }
    )

pd.DataFrame(availability)

In [ ]:
downloaded = {}
standardized = {}
missing_subcategories = []

for spec in FORCING_LIBRARY:
    if spec.subcategory not in climate_catalog:
        missing_subcategories.append(spec.subcategory)
        print(f"[SKIP] {spec.key}: missing catalog entry {spec.subcategory}")
        continue

    ds_path = downloader.download_dataset(
        category=spec.category,
        subcategory=spec.subcategory,
        time_range=TIME_RANGE,
        variables=[spec.source_variable],
    )
    target = STANDARDIZED_DIR.joinpath(spec.output_filename)
    shutil.copy2(ds_path, target)
    downloaded[spec.key] = ds_path
    standardized[spec.key] = target
    print(f"[OK] {spec.key}: {target.name}")

if missing_subcategories:
    print(
        "Missing subcategories skipped:", ", ".join(sorted(set(missing_subcategories)))
    )

pd.DataFrame(
    [
        {
            "forcing_key": k,
            "raw_download": str(downloaded[k]),
            "standardized_file": str(v),
        }
        for k, v in standardized.items()
    ]
)

In [ ]:
# def build_dfs02todfs2_pfs(input_grid: Path, input_ts: Path, output_grid: Path) -> str:
#     return (
#         "[dfs02todfs2]\n"
#         "   CLSID = 'Dfs02ToDfs2.dll'\n"
#         "   TypeName = 'dfs02todfs2'\n"
#         "   [Setup]\n"
#         "      Name = 'Setup Name'\n"
#         f"      InputGrid = |{input_grid}|\n"
#         f"      InputTS = |{input_ts}|\n"
#         f"      OutputGrid = |{output_grid}|\n"
#         "   EndSect  // Setup\n"
#         "EndSect  // dfs02todfs2\n"
#     )


# def run_dfs02todfs2_job(
#     exe_path: Path, pfs_path: Path, input_grid: Path, input_ts: Path, output_grid: Path
# ) -> None:
#     pfs_path.parent.mkdir(parents=True, exist_ok=True)
#     pfs_path.write_text(
#         build_dfs02todfs2_pfs(input_grid, input_ts, output_grid), encoding="utf-8"
#     )

#     if not exe_path.exists():
#         raise FileNotFoundError(f"dfs02todfs2 executable not found: {exe_path}")

#     proc = subprocess.run(
#         [str(exe_path), str(pfs_path)], check=False, capture_output=True, text=True
#     )
#     if proc.returncode != 0:
#         raise RuntimeError(
#             f"dfs02todfs2 failed for {input_ts.name}\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
#         )


# # Edit with your Task3 dfs0 files
# DFS0_INPUTS = {
#     "era5_potential_evapotranspiration": Path(
#         r"C:\\path\\to\\Short Wave Solar Radiation test.dfs0"
#     ),
# # }

# # conversion_rows = []
# # for spec in FORCING_LIBRARY:
# #     if spec.key not in DFS0_INPUTS:
# #         print(f"[SKIP] {spec.key}: no DFS0 input specified")
# #         continue

# #     input_ts = DFS0_INPUTS[spec.key]
# #     output_grid = STANDARDIZED_DIR.joinpath(spec.output_filename)
# #     pfs_file = PFS_WORK_DIR.joinpath(f"{spec.key}_dfs02todfs2.pfs")
# #     run_dfs02todfs2_job(
# #         DFS02TODFS2_EXE, pfs_file, GRID_CODE_DFS2, input_ts, output_grid
# #     )

# #     conversion_rows.append(
# #         {
# #             "forcing_key": spec.key,
# #             "input_dfs0": str(input_ts),
# #             "pfs_file": str(pfs_file),
# #             "output_dfs2": str(output_grid),
# #         }
# #     )

# pd.DataFrame(conversion_rows)